In [2]:
import pickle
import os
import numpy as np
from tqdm import tqdm
from scipy.io import wavfile
from python_speech_features import mfcc
from keras.models import load_model
import pandas as pd
from sklearn.metrics import accuracy_score
from cfg import Config

In [19]:
data_dir = "../../../RAVDESS"
save_dir = "../../dataset/"
clean_dir = save_dir+"clean/"

def build_predictions(audio_dir):
    y_true = []
    y_pred = []
    fn_prob = {}
    
    for fn in tqdm(os.listdir(audio_dir)):
        rate,wav = wavfile.read(os.path.join(audio_dir,fn))
        label = fn2class[fn]
        c = classes.index(label)
        
        y_prob = []
        
        for i in range(0,wav.shape[0]-config.step,config.step):
            sample = wav[i:i+config.step]
            x = mfcc(sample,rate,numcep=config.nfeat,
                     nfilt=config.nfilt,nfft=config.nfft)
            x = (x-config.min)/(config.max - config.min)
            
            if config.mode == "conv":
                x = x.reshape(1,x.shape[0],x.shape[1],1)
            elif config.mode == "time":
                x = np.expand_dims(x_axis=0)
                
            y_hat = model.predict(x)
            y_prob.append(y_hat)
            y_pred.append(np.argmax(y_hat))
            y_true.append(c)
            
        fn_prob[fn] = np.mean(y_prob,axis=0).flatten()
        
    return y_true,y_pred,fn_prob

In [20]:
df = pd.read_csv("audio_data.csv")
classes = list(df["emotion"].unique())
fn2class = dict(zip(df["filename"],df["emotion"]))
p_path = os.path.join("pickles","conv.p")

with open(p_path,"rb") as handle:
    config = pickle.load(handle)
model = load_model("conv.model")
y_true,y_pred,fn_prob = build_predictions(clean_dir)
acc_score = accuracy_score(y_true=y_true,y_pred=y_pred)

y_probs = []

for i,row in df.iterrows():
    y_prob = fn_prob[row["filename"]]
    y_probs.append(y_prob)
    for c,p in zip(classes,y_prob):
        df.at[i,c] = p
y_pred = [classes[np.argmax(y)] for y in y_probs]
df["y_pred"] = y_pred

  0%|                                                                                         | 0/1440 [00:00<?, ?it/s]

1/1 [==============================] - 0s 20ms/step


  0%|                                                                                 | 1/1440 [00:00<20:20,  1.18it/s]

1/1 [==============================] - 0s 19ms/step


  0%|                                                                                 | 2/1440 [00:01<20:45,  1.15it/s]

...

100%|█████████████████████████████████████████████████████████████████████████████▉| 1438/1440 [31:06<00:02,  1.14s/it]

1/1 [==============================] - 0s 21ms/step


100%|█████████████████████████████████████████████████████████████████████████████▉| 1439/1440 [31:07<00:01,  1.04s/it]

1/1 [==============================] - 0s 22ms/step


100%|██████████████████████████████████████████████████████████████████████████████| 1440/1440 [31:08<00:00,  1.30s/it]


In [22]:
df.to_csv("audio_data2.csv")

In [7]:
pd.read_csv("audio_data2.csv")[pd.read_csv("audio_data2.csv").columns[:-10:-1]]

,y_pred,surprised,disgust,fearful,angry,sad,happy,calm,neutral
0,neutral,0.022398,0.041943,0.032759,0.024657,0.143080,0.033109,0.237801,0.464253
1,neutral,0.046731,0.054599,0.048377,0.039293,0.144163,0.050274,0.279448,0.337114
2,neutral,0.030366,0.031807,0.042911,0.008527,0.144498,0.051433,0.323141,0.367316
3,neutral,0.044267,0.034211,0.047515,0.011061,0.158168,0.045996,0.246355,0.412427
4,calm,0.014761,0.058121,0.023932,0.011824,0.119129,0.020906,0.507941,0.243386
...,...,...,...,...,...,...,...,...,...
1435,surprised,0.358825,0.107195,0.131204,0.099809,0.096432,0.131663,0.032937,0.041936
1436,disgust,0.238784,0.255096,0.139608,0.104361,0.100977,0.088155,0.035758,0.037261
1437,surprised,0.291090,0.178098,0.147504,0.098716,0.101783,0.126538,0.031120,0.025151
1438,surprised,0.290136,0.127952,0.219530,0.113137,0.055885,0.099687,0.041905,0.051769
